This file reads RHINO output data (pkl files) and writes them in an openPMD series  
Uses ADIOS2 as a backend (.bp5)  
We should use variable-based iteration encoding, but needs improvements  
For now we save all data at iteration zero  
Stores data in particle records (i.e. DataFrame-like)  
Data is stored in a shared folder on NERSC `/global/cfs/cdirs/m3239/2026_FES-AmSC/data/rhino`

In [ ]:
import openpmd_api as io
import numpy as np
import pandas as pd
from tqdm import tqdm
import sys
import networkx as nx

Open points:
* allow `/` in openPMD attributes
* allow non-numerical data in openPMD particles?
* make D and T records the same length, i.e. pad D with zeros in the subsystems with no D
* save universal subsystems information as series attributes (not as a record) 

In [ ]:
# Configuration
OUTPUT_PATH = "/global/cfs/cdirs/m3239/aforment/IFE_AmSC/RHINO/scripts/tmp/rhino_particles.bp5"

RHINO_PATH = "/global/cfs/cdirs/m3239/2026_FES-AmSC/data/rhino/more_data_runs/" 
DATA_PATH  = f"{RHINO_PATH}/TBR/Data"
INPUT_PATH = f"{RHINO_PATH}/makeJSON.py"

sys.path.append(RHINO_PATH)
from makeJSON import InputFile

SECONDS_PER_DAY = 86400.0

In [ ]:
# Load RHINO Data: time-series and steady-state data
# Note 1: do the filenames include any information?
# Note 2: it would be nice if the structure were DATA_PATH/PREFIX/T.pkl, etc.
PREFIX="12-35-04"
INFIX ="IFE_AmSC_500MW_FuelCycle"
T_ts_df = pd.read_pickle(f"{DATA_PATH}/{PREFIX}_{INFIX}_T.pkl")
D_ts_df = pd.read_pickle(f"{DATA_PATH}/{PREFIX}_{INFIX}_D.pkl")
T_ss_df = pd.read_pickle(f"{DATA_PATH}/{PREFIX}_{INFIX}_T_SteadyState.pkl")
D_ss_df = pd.read_pickle(f"{DATA_PATH}/{PREFIX}_{INFIX}_D_SteadyState.pkl")
# Load RHINO Metafile
# Note: no infix in the metafile name 
meta_df = pd.read_pickle(f"{DATA_PATH}/{PREFIX}_IFE_meta.pkl")

In [ ]:
# Convert metadata to dictionary
meta = meta_df[0].to_dict()
for k, v in list(meta.items()):
    if isinstance(v, np.generic):
        meta[k] = v.item()

# We could also import directly as dictionary 
#import pickle 
#with open(f"{DATA_PATH}/{PREFIX}_IFE_meta.pkl", 'rb') as file:
#    meta = pickle.load(file)

# Get timestep
dt = float(meta["dt"]) if "dt" in meta else None
if dt is None:
    raise ValueError("dt is required for time-series data")   

# Get simulation time 
endtime = float(meta["calc_length"])
times = np.arange(0, endtime+dt, dt)
nt = len(times)

In [ ]:
# Extract data from input file
my_inputs = {}
my_inputs["Tritium"] = {}

for k,v in InputFile['Systems_T'].items():
    my_inputs["Tritium"][v[0]] = {"id": k, 
                                  "processing time": v[1], 
                                  "nonradioactive loss fraction": v[2], 
                                  "fractional inflows": v[3], 
                                  "initial mass": v[4], 
                                  "source": v[5], 
                                  "injectors": v[6], 
                                  "label": v[7]}

my_inputs["Deuterium"] = {}

for k,v in InputFile['Systems_D'].items():
    my_inputs["Deuterium"][v[0]] = {"id": k, 
                                  "processing time": v[1], 
                                  "nonradioactive loss fraction": v[2], 
                                  "fractional inflows": v[3], 
                                  "initial mass": v[4], 
                                  "source": v[5], 
                                  "injectors": v[6], 
                                  "label": v[7]}

all_subsystems_names = np.union1d(list(my_inputs["Tritium"]), list(my_inputs["Deuterium"]))

In [ ]:
# Extract data from pkl files 
T_ts = np.ascontiguousarray(T_ts_df.to_numpy(dtype=np.float64))
nSubsystemsT, Nt = T_ts.shape
T_ss = T_ss_df.iloc[:, 0].to_numpy(dtype=np.float64)

D_ts = np.ascontiguousarray(D_ts_df.to_numpy(dtype=np.float64))
nSubsystemsD, Nt = D_ts.shape
D_ss = D_ss_df.iloc[:, 0].to_numpy(dtype=np.float64)

my_inventory = {}
my_inventory["Tritium"]   = {"data_ts": T_ts, "data_ss": T_ss}
my_inventory["Deuterium"] = {"data_ts": D_ts, "data_ss": D_ss}

In [ ]:
# Create Series
adios2_cfg = r'''
{
  "iteration_encoding": "variable_based",
  "adios2": {
    "modifiable_attributes": false,
    "use_group_table": false,
    "engine": {
      "type": "bp5",
      "parameters": {
        "StatsLevel": "1",
        "AsyncWrite": "guided"
      }
    }
  }
}
'''
OUTPUT_PATH = "/global/cfs/cdirs/m3239/aforment/IFE_AmSC/RHINO/scripts/tmp/rhino_particles.bp5"
series = io.Series(OUTPUT_PATH, io.Access_Type.create_linear, adios2_cfg)
print("Writing RHINO data in particle representation...")

### Series attributes

In [ ]:
# =================
# Software Metadata
series.set_attribute("software:softwareName", "RHINO")
series.set_attribute("software:softwareDescription", "RHINO: Fusion Pilot Plant fuel cycle simulation")
series.set_attribute("software:softwareVersion", "1.0")
# Placeholder for additional attributes under software/
series.set_attribute("software:versionControlSoftware", "")
series.set_attribute("software:softwareCommit", "")
series.set_attribute("software:softwareDocumentation", "")

In [ ]:
# ===================
# Provenance metadata
series.set_attribute("provenance:author", "Holly Flynn")
series.set_attribute("provenance:authorAffiliation", "Savannah River National Laboratory")
series.set_attribute("provenance:authorEmail", "Holly.Flynn@srnl.doe.gov")
series.set_attribute("provenance:creationDate", "2025-12-16")
# Placeholder for additional attributes under provenance/
series.set_attribute("provenance:inputDirectory", " ")
series.set_attribute("provenance:inputFiles", INPUT_PATH)
series.set_attribute("provenance:originalDataDirectory", "") #?
series.set_attribute("provenance:originalDataFiles", "") #?

In [ ]:
# ==============================
# System metadata (placeholders)
series.set_attribute("system:systemIP", "")
series.set_attribute("system:systemDescription", "")

In [ ]:
# ==============================
# How shoud we call this set of metadata? inputs? rhino? application? 
# General inputs (common to D and T) 
series.set_attribute("Tritium Breeding Ratio", InputFile["System Inputs"]["TBR"])
series.set_attribute("Required Tritium Breeding Ratio", InputFile["System Inputs"]["TBRr"])
series.set_attribute("Burn fraction", InputFile["System Inputs"]["beta"])
series.set_attribute("Fueling efficiency", InputFile["System Inputs"]["eta"])
series.set_attribute("Tritium burned per day", InputFile["System Inputs"]["Ndotminus"])
series.set_attribute("Power output in MW", InputFile["System Inputs"]["MW"])
series.set_attribute("Starting inventory", InputFile["System Inputs"]["I0_SD"])

In [ ]:
# =====================================
# Application metadata (RHINO-specific)
# Note that the subsystems ids are not the same for T and D
# -> these attributes should be species-specific and go under particles[name]
# I still keep a list of all subsystem names here, but I am not sure it makes sense
series.set_attribute("subsystems:description", "Subsystems of the pilot plant fuel cycle")
series.set_attribute("subsystems:names", all_subsystems_names.tolist())

In [ ]:
# ==============================
# openPMD-ready attributes (some are duplicates) 
series.particles_path = "inventory"
series.set_attribute("software", "RHINO")
series.set_attribute("softwareVersion", "1.0")
series.set_attribute("softwareDescription", "RHINO: Fusion Pilot Plant fuel cycle simulation")
series.set_attribute("author", "Holly Flynn")
series.set_attribute("authorAffiliation", "Savannah River National Laboratory")
series.set_attribute("authorEmail", "Holly.Flynn@srnl.doe.gov")

### Iteration(s)

In [ ]:
it = series.snapshots()[0]
it.time = 0.0
it.dt = float(dt)
it.time_unit_SI = SECONDS_PER_DAY

### Species

In [ ]:
# times
# it.particles["electrons"]["momentum"]["x"]
species = it.particles["times"]  
species.set_attribute("description", "Times")

record = species["data"]
record.unit_dimension =  {io.Unit_Dimension.T: 1}
record.unit_SI = SECONDS_PER_DAY

data = np.ascontiguousarray(times).copy()
dataset = io.Dataset(times.dtype, times.shape)

component = record[io.Record_Component.SCALAR]
component = record[io.Record_Component.SCALAR]
component.reset_dataset(dataset)
component.store_chunk(data)

In [ ]:
def write_species(name, data_ts, data_ss):

    pt = it.particles[name]

    pt.set_attribute("description", "Inventory across subsystems for species " + name)
    pt.set_attribute("timeAxis", 1)  
    pt.set_attribute("subsystemsAxis", 0)
    
    record = pt["subsystems"]
    
    for k,v in my_inputs[name].items():
        component = record[k]

        # I save the id as if it was the data
        # That's because a record must have data (I think)
        # Otherwise the attributes are not saved (check!)
        for kk, vv in my_inputs[name][k].items():
            if "id" not in kk:
                component.set_attribute(kk, vv)
            else: 
                data = np.ascontiguousarray(vv).copy()
                component.reset_dataset(io.Dataset(data.dtype, data.shape))
                component.store_chunk(data)

    # time-series inventory data (copy to ensure writable for store_chunk)
    data_arr = np.ascontiguousarray(data_ts).copy()
    inv_rec = pt["mass"][io.Record_Component.SCALAR]
    inv_rec.reset_dataset(io.Dataset(data_arr.dtype, data_arr.shape))
    inv_rec.store_chunk(data_arr)

    pt["mass"].unit_dimension = {io.Unit_Dimension.M: 1}
    inv_rec.unit_SI = 1e-3
    
    # steady-state inventory data (copy to ensure writable for store_chunk)
    ss_arr = np.ascontiguousarray(data_ss).copy()
    ss_rec = pt["mass_steady"][io.Record_Component.SCALAR]
    ss_rec.reset_dataset(io.Dataset(ss_arr.dtype, ss_arr.shape))
    ss_rec.store_chunk(ss_arr)

    pt["mass_steady"].unit_dimension = {io.Unit_Dimension.M: 1}
    ss_rec.unit_SI = 1e-3

write_species("Tritium", T_ts, T_ss)
write_species("Deuterium", D_ts, D_ss)

it.close()
series.close()

print("RHINO data written to ADIOS-OpenPMD in particle representation.")
print("Output:", OUTPUT_PATH)